# 01. Exploratory Data Analysis: Multi-Spectral INSAT-3D Satellite Imagery

This notebook demonstrates:
1. Ingestion of INSAT-3D / 3DR Imager L1B radiometric channels (TIR1: 10.8µm, WV: 6.7µm, VIS: 0.65µm, TIR2: 12.0µm)
2. Conversion of raw digital counts to calibrated Brightness Temperatures ($T_b$) in Kelvin
3. Radial cloud-top temperature profiles through the cyclone eye and eyewall
4. Climatological analysis of historical cyclones across the Bay of Bengal and Arabian Sea

In [ ]:
import os
import sys
import datetime
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root is in python path
sys.path.append(os.path.abspath('..'))

from src.data_pipeline.fetch_mosdac_insat import MosdacInsatFetcher
from src.data_pipeline.preprocessor import MultimodalPreprocessor

sns.set_theme(style="darkgrid")
plt.rcParams["figure.figsize"] = (12, 6)

## 1. Synthesize / Ingest Calibrated Multi-Channel Satellite Tensor

In [ ]:
fetcher = MosdacInsatFetcher()
now = datetime.datetime.now(datetime.timezone.utc)

# Synthesize calibrated 4-channel tensor for Very Severe Cyclonic Storm in Bay of Bengal
sat_tensor = fetcher.fetch_or_synthesize_raster(
    timestamp=now,
    center_lat=16.2,
    center_lon=88.5,
    size=(256, 256),
    storm_intensity_knots=90.0
)

print(f"Satellite Array Shape: {sat_tensor.shape} [Channels, Height, Width]")
print(f"TIR1 Range: {sat_tensor[0].min():.1f} K to {sat_tensor[0].max():.1f} K")
print(f"WV Range: {sat_tensor[1].min():.1f} K to {sat_tensor[1].max():.1f} K")

## 2. Multi-Spectral Band Visualization

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

im0 = axes[0].imshow(sat_tensor[0], cmap='inferno_r')
axes[0].set_title('TIR1 (10.8 µm) - Cloud Top Temp (K)')
plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

im1 = axes[1].imshow(sat_tensor[1], cmap='YlGnBu_r')
axes[1].set_title('Water Vapor (6.7 µm) - Moisture (K)')
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

im2 = axes[2].imshow(sat_tensor[2], cmap='gray')
axes[2].set_title('Visible (0.65 µm) - Albedo')
plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

im3 = axes[3].imshow(sat_tensor[3], cmap='magma_r')
axes[3].set_title('TIR2 (12.0 µm) - Split Window (K)')
plt.colorbar(im3, ax=axes[3], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

## 3. Eyewall Brightness Temperature Profile & Subsidence Eye Signature

In [ ]:
center_y = sat_tensor.shape[1] // 2
cross_section_tir1 = sat_tensor[0, center_y, :]
cross_section_wv = sat_tensor[1, center_y, :]

plt.figure(figsize=(10, 4))
plt.plot(cross_section_tir1, label='TIR1 (10.8 µm)', color='crimson', lw=2)
plt.plot(cross_section_wv, label='Water Vapor (6.7 µm)', color='dodgerblue', lw=2, linestyle='--')
plt.axvline(x=128, color='gold', linestyle=':', label='Cyclone Eye Center')
plt.title('Vortex Cross-Sectional Brightness Temperature Profile')
plt.xlabel('Radial Pixel Distance (W-E Axis)')
plt.ylabel('Brightness Temperature (K)')
plt.legend()
plt.show()